# 🛡️ Email Phishing Classifier — Fine-Tune DistilBERT

This notebook fine-tunes **DistilBERT** on your phishing email datasets to build a binary classifier:
- **0 = Legitimate/Safe**
- **1 = Phishing/Malicious**

## Instructions
1. Upload your CSV files from the `archive/` folder when prompted
2. Run all cells top-to-bottom
3. Download the `phishing_model.zip` at the end
4. Unzip into `email-analyser/models/phishing-distilbert/`

**Runtime → Change runtime type → T4 GPU** (free tier)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn torch

## Step 2 — Upload Your Datasets

Upload these files from your `archive/` folder:
- `phishing_email.csv` (primary — has `text_combined`, `label`)
- `Enron.csv` (has `subject`, `body`, `label`)
- `CEAS_08.csv` (has `subject`, `body`, `label`)
- `SpamAssasin.csv` (has `subject`, `body`, `label`)
- `Nazario.csv` (has `subject`, `body`, `label`)
- `Ling.csv` (has `subject`, `body`, `label`)
- `Nigerian_Fraud.csv` (has `subject`, `body`, `label`)

In [ ]:
from google.colab import files
import os

print("📂 Upload ALL CSV files from your archive/ folder...")
print("   Select all 7 CSV files at once (multi-select with Ctrl+Click)\n")
uploaded = files.upload()
print(f"\n✅ Uploaded {len(uploaded)} files: {list(uploaded.keys())}")

## Step 3 — Load & Merge All Datasets

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

all_dataframes = []

# ── Dataset 1: phishing_email.csv (text_combined, label) ──
if os.path.exists('phishing_email.csv'):
    df1 = pd.read_csv('phishing_email.csv', usecols=['text_combined', 'label'])
    df1 = df1.rename(columns={'text_combined': 'text'})
    df1['source'] = 'phishing_email'
    all_dataframes.append(df1)
    print(f"✅ phishing_email.csv: {len(df1)} rows | Labels: {df1['label'].value_counts().to_dict()}")

# ── Datasets with (subject, body, label) format ──
datasets_with_body = [
    'Enron.csv', 'CEAS_08.csv', 'SpamAssasin.csv',
    'Nazario.csv', 'Ling.csv', 'Nigerian_Fraud.csv'
]

for fname in datasets_with_body:
    if os.path.exists(fname):
        try:
            df = pd.read_csv(fname, on_bad_lines='skip', engine='python')
            # Combine subject + body into one text field
            if 'subject' in df.columns and 'body' in df.columns:
                df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
            elif 'body' in df.columns:
                df['text'] = df['body'].fillna('')
            elif 'text_combined' in df.columns:
                df['text'] = df['text_combined'].fillna('')
            else:
                print(f"⚠️  {fname}: No text column found, skipping. Columns: {list(df.columns)}")
                continue

            # Ensure label column exists and is numeric
            if 'label' in df.columns:
                df['label'] = pd.to_numeric(df['label'], errors='coerce')
                df = df.dropna(subset=['label'])
                df['label'] = df['label'].astype(int)
            else:
                print(f"⚠️  {fname}: No label column, skipping.")
                continue

            df = df[['text', 'label']].copy()
            df['source'] = fname
            all_dataframes.append(df)
            print(f"✅ {fname}: {len(df)} rows | Labels: {df['label'].value_counts().to_dict()}")
        except Exception as e:
            print(f"❌ {fname}: Error loading — {e}")

# ── Merge everything ──
combined = pd.concat(all_dataframes, ignore_index=True)
print(f"\n{'='*50}")
print(f"📊 COMBINED DATASET: {len(combined)} total rows")
print(f"   Label 0 (Legitimate): {(combined['label']==0).sum()}")
print(f"   Label 1 (Phishing):   {(combined['label']==1).sum()}")
print(f"   Sources: {combined['source'].nunique()} datasets")

## Step 4 — Clean & Prepare Text

In [ ]:
import re

def clean_email_text(text):
    """Clean email text for model training."""
    if not isinstance(text, str):
        return ""
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove URLs (keep domain for context)
    text = re.sub(r'https?://\S+', '[URL]', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', '[EMAIL]', text)
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Lowercase
    text = text.lower()
    # Truncate very long texts (DistilBERT max is 512 tokens ≈ 1500 chars)
    if len(text) > 2000:
        text = text[:2000]
    return text

print("🧹 Cleaning text...")
combined['text'] = combined['text'].apply(clean_email_text)

# Remove empty/very short texts
combined = combined[combined['text'].str.len() > 20].reset_index(drop=True)

# Ensure binary labels only (0 or 1)
combined = combined[combined['label'].isin([0, 1])].reset_index(drop=True)

# Remove duplicates
before = len(combined)
combined = combined.drop_duplicates(subset=['text']).reset_index(drop=True)
after = len(combined)

print(f"✅ Cleaned: {after} rows (removed {before - after} duplicates)")
print(f"   Label 0 (Legitimate): {(combined['label']==0).sum()}")
print(f"   Label 1 (Phishing):   {(combined['label']==1).sum()}")

# Show sample
print("\n📧 Sample legitimate email:")
print(combined[combined['label']==0]['text'].iloc[0][:200])
print("\n📧 Sample phishing email:")
print(combined[combined['label']==1]['text'].iloc[0][:200])

## Step 5 — Balance the Dataset

In [ ]:
# Balance classes by undersampling the majority class
# This prevents the model from being biased toward one class

legit = combined[combined['label'] == 0]
phish = combined[combined['label'] == 1]

min_class_size = min(len(legit), len(phish))
# Cap at 25000 per class to keep training time reasonable on free Colab GPU
samples_per_class = min(min_class_size, 25000)

legit_balanced = legit.sample(n=samples_per_class, random_state=42)
phish_balanced = phish.sample(n=samples_per_class, random_state=42)

balanced = pd.concat([legit_balanced, phish_balanced]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Balanced dataset: {len(balanced)} rows")
print(f"   Label 0 (Legitimate): {(balanced['label']==0).sum()}")
print(f"   Label 1 (Phishing):   {(balanced['label']==1).sum()}")

## Step 6 — Train/Validation/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# 80% train, 10% validation, 10% test
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    balanced['text'].tolist(),
    balanced['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=balanced['label'].tolist()
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print(f"📊 Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

## Step 7 — Tokenize with DistilBERT Tokenizer

In [ ]:
from transformers import DistilBertTokenizerFast
import torch
from torch.utils.data import Dataset

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class EmailDataset(Dataset):
    """Custom PyTorch dataset for email classification."""
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

print("🔤 Tokenizing datasets...")
train_dataset = EmailDataset(train_texts, train_labels, tokenizer)
val_dataset = EmailDataset(val_texts, val_labels, tokenizer)
test_dataset = EmailDataset(test_texts, test_labels, tokenizer)
print(f"✅ Tokenization complete")

## Step 8 — Fine-Tune DistilBERT

This will take ~15-25 minutes on T4 GPU (Colab free tier).

In [ ]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Load pre-trained DistilBERT with a classification head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

# Define evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Training configuration
training_args = TrainingArguments(
    output_dir='./training_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',  # Disable wandb
    fp16=True,  # Mixed precision for faster training on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting fine-tuning... This takes ~15-25 min on T4 GPU")
print("   Watch the loss decrease and F1 score increase each epoch\n")
trainer.train()

## Step 9 — Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Run evaluation on test set
print("📊 Evaluating on test set...\n")
results = trainer.evaluate(test_dataset)

print(f"{'='*50}")
print(f"📈 TEST SET RESULTS")
print(f"{'='*50}")
print(f"   Accuracy:  {results['eval_accuracy']:.4f}")
print(f"   F1 Score:  {results['eval_f1']:.4f}")
print(f"   Precision: {results['eval_precision']:.4f}")
print(f"   Recall:    {results['eval_recall']:.4f}")
print(f"{'='*50}")

# Detailed classification report
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print("\n📋 CLASSIFICATION REPORT:")
print(classification_report(
    test_labels, preds,
    target_names=['Legitimate', 'Phishing']
))

print("📋 CONFUSION MATRIX:")
cm = confusion_matrix(test_labels, preds)
print(f"                 Predicted")
print(f"                 Legit  Phish")
print(f"   Actual Legit  {cm[0][0]:5d}  {cm[0][1]:5d}")
print(f"   Actual Phish  {cm[1][0]:5d}  {cm[1][1]:5d}")

## Step 10 — Quick Test with Sample Emails

In [ ]:
from transformers import pipeline

# Create inference pipeline
classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Test samples
test_samples = [
    {
        'name': '🟢 Clean business email',
        'text': 'Hi team, please find attached the Q3 financial report. The board meeting is scheduled for next Thursday at 2pm in conference room B. Let me know if you have any questions. Best regards, Sarah'
    },
    {
        'name': '🔴 Phishing — credential harvest',
        'text': 'URGENT: Your account has been compromised. Click here immediately to verify your identity and reset your password or your account will be permanently suspended within 24 hours. Verify now at http://secure-login.account-verify.com/reset'
    },
    {
        'name': '🔴 Phishing — Nigerian scam',
        'text': 'Dear Friend, I am Prince Abubakar from Nigeria. I have $15,000,000 USD that I need your help to transfer. You will receive 30% of the total amount. Please send your bank details and a small processing fee of $500 to begin the transfer.'
    },
    {
        'name': '🟢 Clean newsletter',
        'text': 'Weekly Tech Digest: This week in technology — Apple announces new MacBook Pro, Google releases Android update, Microsoft patches security vulnerability in Windows. Unsubscribe from this newsletter at any time.'
    }
]

print("🧪 TESTING MODEL WITH SAMPLE EMAILS")
print("=" * 60)
for sample in test_samples:
    result = classifier(sample['text'])[0]
    label = 'PHISHING' if result['label'] == 'LABEL_1' else 'LEGITIMATE'
    confidence = result['score']
    icon = '🚨' if label == 'PHISHING' else '✅'
    print(f"\n{sample['name']}")
    print(f"   → {icon} {label} (confidence: {confidence:.2%})")

## Step 11 — Save & Download Model

This saves the model and tokenizer, then zips it for download.
After downloading, unzip to: `email-analyser/models/phishing-distilbert/`

In [ ]:
import shutil
import json

SAVE_DIR = './phishing-distilbert'

# Save model and tokenizer
print("💾 Saving model and tokenizer...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save training metadata
metadata = {
    'model_name': 'phishing-distilbert',
    'base_model': 'distilbert-base-uncased',
    'task': 'binary_classification',
    'labels': {0: 'legitimate', 1: 'phishing'},
    'training_samples': len(train_texts),
    'test_accuracy': results['eval_accuracy'],
    'test_f1': results['eval_f1'],
    'test_precision': results['eval_precision'],
    'test_recall': results['eval_recall'],
    'max_length': 512,
    'epochs': 3,
    'datasets_used': list(combined['source'].unique())
}

with open(f'{SAVE_DIR}/training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Model saved to {SAVE_DIR}/")
print(f"   Files: {os.listdir(SAVE_DIR)}")

# Zip for download
print("\n📦 Creating zip file...")
shutil.make_archive('phishing_model', 'zip', '.', SAVE_DIR)
print(f"✅ Created phishing_model.zip ({os.path.getsize('phishing_model.zip') / 1e6:.1f} MB)")

# Download
print("\n⬇️  Downloading... (check your browser downloads)")
files.download('phishing_model.zip')

## ✅ Done!

### Next Steps:
1. The `phishing_model.zip` should be downloading to your computer
2. Unzip it
3. Move the `phishing-distilbert/` folder to: `email-analyser/models/phishing-distilbert/`
4. The folder should contain these files:
   - `config.json`
   - `model.safetensors`
   - `tokenizer.json`
   - `tokenizer_config.json`
   - `vocab.txt`
   - `special_tokens_map.json`
   - `training_metadata.json`

The main project code (`ml_classifier.py`) will automatically load from this path.